# SC 612 104 Essential Data Science
## คาบที่ 9: groupby, agg, sort_values, value_counts, pivot_table, merge/join, concat, apply + lambda

**ผู้สอน:** อ.ดร.พิชญา วิรัชโชติเสถียร (Pitchaya Wiratchotisatian)
**ภาควิชาสถิติ คณะวิทยาศาสตร์ มหาวิทยาลัยขอนแก่น**

---

### เนื้อหาในคาบนี้

**ส่วนที่ 1: การสรุปและเรียงข้อมูล**
1. `groupby()` — จัดกลุ่มข้อมูล
2. `agg()` — สรุปสถิติหลายแบบพร้อมกัน
3. `sort_values()` — เรียงลำดับข้อมูล
4. `value_counts()` — นับความถี่ของค่า
5. `pivot_table()` เบื้องต้น — ตารางสรุปแบบ 2 มิติ

**ส่วนที่ 2: การรวมข้อมูลจากหลายตาราง**\
6. `merge()` / `join()` — รวม DataFrame ตามคีย์\
7. `concat()` — ต่อ DataFrame เข้าด้วยกัน

**ส่วนที่ 3: การประมวลผลแบบกำหนดเอง**\
8. `apply()` + `lambda` — เขียนฟังก์ชันกำหนดเองไปใช้กับข้อมูล

**แบบฝึกหัดท้ายคาบ**

> 📌 **ก่อนเริ่ม:** Notebook นี้ต่อเนื่องจาก `SC612104_Session8_Pandas_Basics.ipynb` (Series, DataFrame, read_csv, info, describe, loc/iloc, boolean indexing) — ถ้ายังไม่คล่องคาบที่แล้ว แนะนำให้กลับไปทวนก่อน

> 📁 **ไฟล์ที่ต้องใช้ในคาบนี้:**
> - `world_weather_sample.csv` — ข้อมูลอากาศจำลอง (ไฟล์เดิมจาก Session 8)
> - `city_info.csv` — ข้อมูลเมือง (ประชากร, timezone) **ไฟล์สะอาด ไม่มี missing values หรือ outliers** — ใช้สอน merge/join
> - `region_info.csv` — ข้อมูลภูมิภาค (continent code, GDP growth) **ไฟล์สะอาดเช่นกัน** — ใช้สอน merge หลายตาราง
>
> อัปโหลดทั้ง 3 ไฟล์เข้า Colab ก่อนเริ่ม (ลากไฟล์ลงในแถบ Files ทางซ้าย หรือใช้โค้ดอัปโหลดด้านล่าง)


In [ ]:
# ถ้าใช้ Google Colab และยังไม่ได้อัปโหลดไฟล์ ให้รันเซลล์นี้เพื่ออัปโหลดทั้ง 3 ไฟล์พร้อมกัน
# (เลือกได้หลายไฟล์ตอนที่หน้าต่างอัปโหลดเปิดขึ้นมา)

try:
    from google.colab import files
    uploaded = files.upload()   # เลือก world_weather_sample.csv, city_info.csv, region_info.csv
except ImportError:
    print("ไม่ได้รันบน Colab - ข้ามขั้นตอนนี้ (ตรวจสอบว่าไฟล์ CSV ทั้ง 3 อยู่ในโฟลเดอร์เดียวกับ Notebook นี้)")

Saving city_info.csv to city_info.csv


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("world_weather_sample.csv")
df["date"] = pd.to_datetime(df["date"])   # แปลงเป็น datetime เหมือนที่เรียนมาจาก Session 8
print("โหลดข้อมูลสำเร็จ! ขนาด:", df.shape)
df.head()

โหลดข้อมูลสำเร็จ! ขนาด: (2075, 10)


,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
0,983,2025-03-24,New York,USA,North America,15.7,72.6,6.3,15.9,1005.4
1,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1
2,1022,2025-02-01,Los Angeles,USA,North America,20.6,46.8,1.2,8.6,1013.5
3,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1
4,1834,2025-02-03,Cape Town,South Africa,Africa,19.1,55.0,0.0,16.6,1015.1


---
## 1. `groupby()` — จัดกลุ่มข้อมูล

นี่คือคำสั่งที่ทรงพลังที่สุดอันหนึ่งของ pandas — ใช้ตอบคำถามแบบ **"แยกตาม..."** เช่น *"อุณหภูมิเฉลี่ยแยกตามเมืองเป็นเท่าไหร่?"*

**แนวคิดหลักของ `groupby()` มี 3 ขั้นตอน (เรียกว่า Split-Apply-Combine):**
1. **Split** — แบ่งข้อมูลเป็นกลุ่มๆ ตามคอลัมน์ที่กำหนด
2. **Apply** — ทำการคำนวณบางอย่างกับแต่ละกลุ่ม (เช่น หาค่าเฉลี่ย)
3. **Combine** — รวมผลลัพธ์ของทุกกลุ่มกลับมาเป็นตารางเดียว

เปรียบเทียบกับสิ่งที่เรียนมา: คล้ายกับการใช้ `for` loop วน list ของ object แล้วจัดกลุ่มด้วย dict — แต่ `groupby()` ทำทั้งหมดให้อัตโนมัติในคำสั่งเดียว และเร็วกว่ามาก


In [4]:
df.groupby("region")[["temperature_c", "humidity_pct",	"rainfall_mm",	"wind_speed_kmh",	"pressure_hpa"]].mean().head(10)


,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
region,,,,,
Africa,26.766479,63.613372,4.018234,15.004213,1013.198889
Asia,27.920223,66.155641,5.899053,15.073496,1013.590304
Europe,14.712079,71.431714,3.198006,14.896884,1012.635574
North America,16.751541,60.379825,3.186494,14.398315,1013.064507
Oceania,19.076111,71.234104,3.972674,14.423429,1013.302825
South America,21.169030,71.692720,3.454753,14.443130,1012.770896


In [ ]:
# อุณหภูมิเฉลี่ยแยกตามเมือง
city_avg_temp = df.groupby("city")["temperature_c"].mean()
print(city_avg_temp)
print(type(city_avg_temp))   # ได้ Series กลับมา (index คือชื่อเมือง)

city
Auckland        17.804444
Bangkok         32.453846
Beijing         15.598876
Berlin          12.383146
Buenos Aires    19.171910
Cairo           26.089888
Cape Town       30.162921
Chiang Mai      29.680220
Lagos           29.494253
Lima            21.490000
London          13.490909
Los Angeles     20.657303
Mexico City     19.187778
Moscow           7.233333
Mumbai          30.872222
Nairobi         21.440000
New York        15.476667
Paris           25.811236
Sao Paulo       22.841573
Singapore       40.083333
Sydney          20.347778
Tokyo           18.414773
Toronto         11.613636
Name: temperature_c, dtype: float64
<class 'pandas.core.series.Series'>


### 1.1 `groupby()` กับหลายคอลัมน์พร้อมกัน

จัดกลุ่มตามมากกว่า 1 คอลัมน์ได้ — ส่ง list ของชื่อคอลัมน์เข้าไป (เชื่อมกับ list ที่เรียนมา)


In [ ]:
# อุณหภูมิเฉลี่ย แยกตามภูมิภาคและเมือง (2 ระดับ)
region_city_avg = df.groupby(["region", "country"])["temperature_c"].mean()
print(region_city_avg)

region         country     
Africa         Egypt           26.089888
               KENYA           19.800000
               Kenya           21.458427
               Nigeria         29.494253
               South Africa    30.162921
Asia           China           15.598876
               India           30.872222
               JAPAN           20.200000
               Japan           18.394253
               SINGAPORE       29.300000
               Singapore       40.204494
               Thailand        31.067033
Europe         France          25.811236
               GERMANY         14.700000
               Germany         12.356818
               Russia           7.233333
               UK              13.490909
North America  CANADA          11.600000
               Canada          11.613953
               Mexico          19.187778
               USA             18.052514
Oceania        Australia       20.347778
               NEW ZEALAND     19.800000
               New Zealand   

### 1.2 `groupby()` กับหลายคอลัมน์ผลลัพธ์ (ไม่ใช่แค่คอลัมน์เดียว)

ถ้าไม่เจาะจงคอลัมน์ตอน aggregate จะได้ผลลัพธ์ของทุกคอลัมน์ตัวเลขพร้อมกัน


In [ ]:
# ค่าเฉลี่ยของทุกคอลัมน์ตัวเลข แยกตามเมือง
city_summary = df.groupby("country")[["temperature_c", "humidity_pct", "rainfall_mm"]].mean()
print(city_summary)

              temperature_c  humidity_pct  rainfall_mm
country                                               
ARGENTINA         22.200000     80.300000     9.300000
Argentina         19.137500     68.922353     3.400000
Australia         20.347778     65.575556     3.809195
Brazil            22.841573     71.208046     4.809091
CANADA            11.600000     67.650000     0.000000
Canada            11.613953     65.503529     3.701163
China             15.598876     44.611628     2.842045
Egypt             26.089888     45.726136     1.397727
France            25.811236     70.964773     3.788636
GERMANY           14.700000     79.800000     0.000000
Germany           12.356818     69.667816     3.120930
India             30.872222     70.404545     5.544318
JAPAN             20.200000     65.200000     0.000000
Japan             18.394253     59.771765     4.426437
KENYA             19.800000     59.200000     2.900000
Kenya             21.458427     60.464706     4.510345
Mexico    

### 1.3 ฟังก์ชันสรุปที่ใช้บ่อยกับ `groupby()`

ใช้ฟังก์ชันสถิติที่เรียนมาจาก NumPy/pandas ต่อท้าย `groupby()` ได้เลย: `.mean()`, `.sum()`, `.count()`, `.max()`, `.min()`, `.std()`, `.median()`


In [ ]:
print("ผลรวมฝนตกแต่ละเมือง (รวม 90 วัน):")
print(df.groupby("city")["rainfall_mm"].sum().sort_values(ascending=False))

print("\nจำนวนแถวข้อมูลของแต่ละเมือง (เช็คว่าครบ 90 วันไหม):")
print(df.groupby("city")["temperature_c"].count())   # .count() ไม่นับ missing values!

print("\nจำนวนแถวข้อมูลของแต่ละเมือง (เช็คว่ามีจำนวน 90 วันไหม):")
(df.groupby("city")["temperature_c"].size())

ผลรวมฝนตกแต่ละเมือง (รวม 90 วัน):
city
Singapore       870.4
Bangkok         685.6
Lagos           652.8
Mumbai          487.9
Chiang Mai      435.6
Sao Paulo       423.2
Nairobi         395.3
Tokyo           385.1
Auckland        351.9
Paris           333.4
Sydney          331.4
Toronto         318.3
New York        315.2
Mexico City     305.9
Buenos Aires    305.1
London          273.3
Berlin          268.4
Beijing         250.1
Moscow          247.4
Cape Town       239.3
Lima            180.3
Los Angeles     169.5
Cairo           123.0
Name: rainfall_mm, dtype: float64

จำนวนแถวข้อมูลของแต่ละเมือง (เช็คว่าครบ 90 วันไหม):
city
Auckland        90
Bangkok         91
Beijing         89
Berlin          89
Buenos Aires    89
Cairo           89
Cape Town       89
Chiang Mai      91
Lagos           87
Lima            90
London          88
Los Angeles     89
Mexico City     90
Moscow          90
Mumbai          90
Nairobi         90
New York        90
Paris           89
Sao Paulo       89
Si

,temperature_c
city,
Auckland,90
Bangkok,91
Beijing,90
Berlin,90
Buenos Aires,90
Cairo,91
Cape Town,90
Chiang Mai,91
Lagos,90


In [ ]:
df.groupby("country").count().head(10)

,record_id,date,city,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa
country,,,,,,,,,
ARGENTINA,1,1,1,1,1,1,1,1,1
Argentina,89,89,89,89,88,85,87,87,88
Australia,90,90,90,90,90,90,87,86,88
Brazil,90,90,90,90,89,87,88,88,90
CANADA,2,2,2,2,2,2,2,2,2
Canada,88,88,88,88,86,85,86,87,86
China,90,90,90,90,89,86,88,88,89
Egypt,91,91,91,91,89,88,88,90,91
France,90,90,90,90,89,88,88,88,89


**🔍 สังเกต:** `.count()` ในตัวอย่างข้างบนอาจไม่เท่ากับ 90 ทุกเมือง เพราะ `.count()` **ไม่นับค่าที่เป็น missing (NaN)** — ถ้าอยากนับจำนวนแถวทั้งหมดรวม missing ด้วย ให้ใช้ `.size()` แทน


In [ ]:
print("เทียบ .count() (ไม่รวม missing) กับ .size() (รวมทุกแถว):")
comparison = pd.DataFrame({
    "count": df.groupby("city")["temperature_c"].count(),
    "size": df.groupby("city").size()
})
comparison.head(10)

เทียบ .count() (ไม่รวม missing) กับ .size() (รวมทุกแถว):


,count,size
city,,
Auckland,90,90
Bangkok,91,91
Beijing,89,90
Berlin,89,90
Buenos Aires,89,90
Cairo,89,91
Cape Town,89,90
Chiang Mai,91,91
Lagos,87,90


---
## 2. `agg()` — สรุปสถิติหลายแบบพร้อมกัน

ถ้าอยากได้สถิติ**หลายแบบพร้อมกัน** (เช่น ทั้ง mean และ max และ min ในคำสั่งเดียว) ใช้ `.agg()` แทนการเขียน `.mean()`, `.max()`, `.min()` แยกกัน


In [ ]:
# สรุปหลายสถิติพร้อมกันด้วย list ของชื่อฟังก์ชัน (เชื่อมกับ list ที่เรียนมา)
city_stats = df.groupby("city")["temperature_c"].agg(["mean", "min", "max", "std"])
city_stats.head(10)

,mean,min,max,std
city,,,,
Auckland,17.804444,11.2,25.6,2.799475
Bangkok,32.453846,26.7,39.1,2.480695
Beijing,15.598876,9.2,22.6,2.874792
Berlin,12.383146,6.9,19.4,2.595986
Buenos Aires,19.171910,-88.0,26.4,11.817985
Cairo,26.089888,20.0,33.6,2.733714
Cape Town,30.162921,13.2,999.0,103.899246
Chiang Mai,29.680220,23.2,37.3,2.858019
Lagos,29.494253,21.1,37.2,2.838414


### 2.1 `agg()` กับหลายคอลัมน์ คนละสถิติกัน (ใช้ dict — เชื่อมกับ dict ที่เรียนมา)

ระบุ dict ที่ key คือชื่อคอลัมน์ value คือฟังก์ชันสถิติที่ต้องการ — ยืดหยุ่นกว่าวิธีแรกมาก


In [ ]:
# คนละคอลัมน์ คนละสถิติ: อุณหภูมิเอาเฉลี่ย, ฝนเอาผลรวม, ความเร็วลมเอาค่าสูงสุด
custom_summary = df.groupby("city").agg({
    "temperature_c":[ "mean","median"],
     "rainfall_mm": "sum",
     "wind_speed_kmh": "max",
})
(custom_summary)

temperature_c        rainfall_mm wind_speed_kmh
                      mean median         sum            max
city                                                        
Auckland         17.804444  17.55       351.9           26.0
Bangkok          32.453846  32.50       685.6           29.8
Beijing          15.598876  15.50       250.1           27.6
Berlin           12.383146  12.30       268.4           27.1
Buenos Aires     19.171910  20.70       305.1           31.0
Cairo            26.089888  25.80       123.0           30.6
Cape Town        30.162921  18.90       239.3           31.6
Chiang Mai       29.680220  29.50       435.6           33.5
Lagos            29.494253  29.80       652.8           28.3
Lima             21.490000  21.30       180.3           33.7
London           13.490909  13.50       273.3           28.3
Los Angeles      20.657303  20.70       169.5           25.2
Mexico City      19.187778  19.50       305.9           31.9
Moscow            7.233333   7.30       247.4           25.9
Mumbai           30.872222  30.95       487.9           25.0
Nairobi          21.440000  21.55       395.3           27.2
New York         15.476667  15.60       315.2           27.5
Paris            25.811236  14.90       333.4           30.1
Sao Paulo        22.841573  22.90       423.2           32.6
Singapore        40.083333  30.80       870.4           30.6
Sydney           20.347778  20.40       331.4           28.0
Tokyo            18.414773  18.50       385.1           30.5
Toronto          11.613636  11.85       318.3           30.9

In [ ]:
custom_summary.head(10)
custom_summary.columns
custom_summary.loc[:, ("temperature_c", ("mean","median"))].head(10)


temperature_c       
                      mean median
city                             
Auckland         17.804444  17.55
Bangkok          32.453846  32.50
Beijing          15.598876  15.50
Berlin           12.383146  12.30
Buenos Aires     19.171910  20.70
Cairo            26.089888  25.80
Cape Town        30.162921  18.90
Chiang Mai       29.680220  29.50
Lagos            29.494253  29.80
Lima             21.490000  21.30

### 2.2 `agg()` พร้อมตั้งชื่อคอลัมน์ผลลัพธ์เอง (Named Aggregation)

วิธีที่อ่านง่ายที่สุดและนิยมใช้ในงานจริง — ตั้งชื่อคอลัมน์ผลลัพธ์ได้ตรงๆ ไม่ต้องมาเปลี่ยนชื่อทีหลัง


In [ ]:
summary = df.groupby("city").agg(
    avg_temp=("temperature_c", "mean"),
    total_rain=("rainfall_mm", "sum"),
    max_wind=("wind_speed_kmh", "max"),
    days_recorded=("temperature_c", "count"),
)
(summary.head(10))

,avg_temp,total_rain,max_wind,days_recorded
city,,,,
Auckland,17.804444,351.9,26.0,90
Bangkok,32.453846,685.6,29.8,91
Beijing,15.598876,250.1,27.6,89
Berlin,12.383146,268.4,27.1,89
Buenos Aires,19.171910,305.1,31.0,89
Cairo,26.089888,123.0,30.6,89
Cape Town,30.162921,239.3,31.6,89
Chiang Mai,29.680220,435.6,33.5,91
Lagos,29.494253,652.8,28.3,87


---
## 3. `sort_values()` — เรียงลำดับข้อมูล

ใช้เรียงแถวของ DataFrame ตามค่าในคอลัมน์ที่กำหนด — เทียบกับ `.sort()` ของ list ที่เรียนมา แต่ทำงานกับตารางทั้งตาราง


In [ ]:
# เรียงเมืองตามอุณหภูมิเฉลี่ย จากน้อยไปมาก (default)
city_avg = df.groupby("city")["temperature_c"].mean().reset_index()
city_avg_sorted = city_avg.sort_values("temperature_c")
print(city_avg_sorted.head())

print("\nเรียงจากมากไปน้อย (ascending=False):")
city_avg_sorted_desc = city_avg.sort_values("temperature_c", ascending=False)#ถ้าจะเรียงจากน้อยไปมากต้อง ascending=True ก็ทำได้
print(city_avg_sorted_desc.head())

        city  temperature_c
13    Moscow       7.233333
22   Toronto      11.613636
3     Berlin      12.383146
10    London      13.490909
16  New York      15.476667

เรียงจากมากไปน้อย (ascending=False):
          city  temperature_c
19   Singapore      40.083333
1      Bangkok      32.453846
14      Mumbai      30.872222
6    Cape Town      30.162921
7   Chiang Mai      29.680220


**หมายเหตุ:** `.reset_index()` ใช้แปลงผลลัพธ์จาก `groupby()` (ที่ city กลายเป็น index) กลับเป็น DataFrame ปกติที่มี city เป็นคอลัมน์ธรรมดา — ใช้บ่อยมากหลัง `groupby()` เวลาต้องการเรียงลำดับหรือใช้ `.loc[]`/`.iloc[]` ต่อ

### 3.1 เรียงตามหลายคอลัมน์พร้อมกัน


In [ ]:
# เรียงตาม region ก่อน แล้วเรียงตาม temperature_c ภายในกลุ่มเดียวกัน
sorted_multi = df.sort_values(["region", "temperature_c"], ascending=[True, False])
(sorted_multi[["region", "city", "temperature_c"]].head(10))

,region,city,temperature_c
78,Africa,Cape Town,999.0
1065,Africa,Lagos,37.2
1152,Africa,Lagos,35.8
1620,Africa,Lagos,35.4
122,Africa,Lagos,34.1
418,Africa,Lagos,34.0
94,Africa,Cairo,33.6
277,Africa,Lagos,33.2
21,Africa,Lagos,32.8
1311,Africa,Lagos,32.8


### 3.2 ตัวอย่างที่ใช้บ่อย: หา Top N ด้วย `sort_values()` + `head()`


In [ ]:
# หา 5 วันที่ฝนตกหนักที่สุดในข้อมูลทั้งหมด
top_rainy_days = df.sort_values("rainfall_mm", ascending=False).head(5)
(top_rainy_days[["city", "date", "rainfall_mm"]])


,city,date,rainfall_mm
1676,Singapore,2025-01-07,17.6
1628,Lagos,2025-02-18,17.3
85,Singapore,2025-02-19,17.2
383,Singapore,2025-01-09,17.1
405,Singapore,2025-03-13,16.6


In [ ]:
# pandas มี .nlargest() / .nsmallest() ที่ทำสิ่งเดียวกันแบบสั้นกว่า
print("\nเทียบกับ .nlargest() (ทำสิ่งเดียวกัน เขียนสั้นกว่า):")
(df.nlargest(5, "rainfall_mm")[["city", "date", "rainfall_mm"]])


เทียบกับ .nlargest() (ทำสิ่งเดียวกัน เขียนสั้นกว่า):


,city,date,rainfall_mm
1676,Singapore,2025-01-07,17.6
1628,Lagos,2025-02-18,17.3
85,Singapore,2025-02-19,17.2
383,Singapore,2025-01-09,17.1
405,Singapore,2025-03-13,16.6


---
## 4. `value_counts()` — นับความถี่ของค่า

ใช้นับว่าแต่ละค่าที่ไม่ซ้ำกัน (unique value) ปรากฏกี่ครั้ง — เทียบกับการใช้ `set()` หา unique values แล้ววน loop นับเอง (ที่เรียนมาจาก Collections) แต่ `.value_counts()` ทำให้ในคำสั่งเดียว


In [ ]:
(df["country"].value_counts())

,count
country,
Thailand,182
USA,180
Egypt,91
Russia,91
Australia,90
Mexico,90
China,90
Kenya,90
Nigeria,90


**สังเกต:** ผลลัพธ์เรียงจากมากไปน้อยอัตโนมัติ — เห็นภาพรวมได้ทันทีว่าภูมิภาคไหนมีข้อมูลมาก/น้อย

### 4.1 `value_counts()` แบบเปอร์เซ็นต์


In [ ]:
print("เป็นสัดส่วน (normalize=True):")
print(df["region"].value_counts(normalize=True))   # ได้สัดส่วน 0-1 แทนจำนวนแถวจริง

print("\nคูณ 100 ให้เป็นเปอร์เซ็นต์:")
print((df["region"].value_counts(normalize=True) * 100).round(1))

เป็นสัดส่วน (normalize=True):
region
Asia             0.261205
Africa           0.174458
Europe           0.173976
North America    0.173494
South America    0.130120
Oceania          0.086747
Name: proportion, dtype: float64

คูณ 100 ให้เป็นเปอร์เซ็นต์:
region
Asia             26.1
Africa           17.4
Europe           17.4
North America    17.3
South America    13.0
Oceania           8.7
Name: proportion, dtype: float64


### 4.2 ใช้ `value_counts()` ตรวจสอบความสะอาดของข้อมูล (เชื่อมกับ Session 8)

จากที่รู้ว่าคอลัมน์ `country` มีตัวพิมพ์ไม่สม่ำเสมอ (จาก Session 8) — `value_counts()` ช่วยให้เห็นปัญหานี้ชัดเจน


In [ ]:
print("จำนวนค่าไม่ซ้ำของ country (ตามตัวพิมพ์จริง):")
print(df["country"].value_counts())
print(f"\nจำนวน unique values ทั้งหมด: {df['country'].nunique()}")
print("(ถ้าได้มากกว่า 23 ประเทศ แสดงว่ามีตัวพิมพ์ไม่สม่ำเสมอปนอยู่ เช่น 'Thailand' กับ 'THAILAND' ถูกนับแยกกัน)")

จำนวนค่าไม่ซ้ำของ country (ตามตัวพิมพ์จริง):
country
Thailand        182
USA             180
Egypt            91
Russia           91
Australia        90
Mexico           90
China            90
Kenya            90
Nigeria          90
France           90
South Africa     90
India            90
Brazil           90
UK               90
Peru             90
Japan            89
Argentina        89
New Zealand      89
Singapore        89
Germany          89
Canada           88
CANADA            2
GERMANY           1
SINGAPORE         1
NEW ZEALAND       1
ARGENTINA         1
JAPAN             1
KENYA             1
Name: count, dtype: int64

จำนวน unique values ทั้งหมด: 28
(ถ้าได้มากกว่า 23 ประเทศ แสดงว่ามีตัวพิมพ์ไม่สม่ำเสมอปนอยู่ เช่น 'Thailand' กับ 'THAILAND' ถูกนับแยกกัน)


---
## 5. `pivot_table()` เบื้องต้น — ตารางสรุปแบบ 2 มิติ

`pivot_table()` คือการสรุปข้อมูลแบบ "ตาราง 2 มิติ" — คล้าย Pivot Table ใน Excel มาก โดยกำหนด:
- **index** → ค่าที่จะอยู่เป็น "แถว"
- **columns** → ค่าที่จะอยู่เป็น "คอลัมน์"
- **values** → ค่าที่จะเอามาคำนวณ
- **aggfunc** → ฟังก์ชันสรุปที่ใช้ (default คือ `mean`)

เปรียบเทียบกับ `groupby()`: `groupby()` ให้ผลลัพธ์เป็น "รายการยาวๆ" ส่วน `pivot_table()` จัดเรียงผลลัพธ์ให้เป็น "ตาราง" ที่อ่านง่ายกว่าเมื่อมี 2 มิติที่อยากเปรียบเทียบ


In [ ]:
# สร้างคอลัมน์เดือนจาก date (ใช้ .dt accessor ที่เรียนมาจาก Session 8)
df["month"] = df["date"].dt.month

# pivot table: แถว = region, คอลัมน์ = month, ค่า = อุณหภูมิเฉลี่ย
pivot = df.pivot_table(
    index="region",
    columns="month",
    values="temperature_c",
    aggfunc="mean"
)
(pivot.round(1))

month,1,2,3
region,,,
Africa,22.6,32.6,25.4
Asia,24.2,26.8,32.7
Europe,10.4,12.2,21.4
North America,15.2,17.1,18.0
Oceania,17.4,19.7,20.2
South America,19.2,22.1,22.3


**อ่านผลลัพธ์:** แต่ละแถวคือภูมิภาค แต่ละคอลัมน์คือเดือน (1, 2, 3 = มกราคม, กุมภาพันธ์, มีนาคม) ค่าในตารางคืออุณหภูมิเฉลี่ยของภูมิภาคนั้นในเดือนนั้น — มองเห็นแนวโน้มตามเวลาและภูมิภาคได้ในตารางเดียว ซึ่งยากกว่าถ้าใช้ `groupby()` อย่างเดียว

### 5.1 `pivot_table()` กับ `aggfunc` หลายแบบ


In [ ]:
# ดูทั้งค่าเฉลี่ยและผลรวมพร้อมกัน
pivot_multi = df.pivot_table(
    index="region",
    columns="month",
    values="rainfall_mm",
    aggfunc=["mean", "sum","median"]
)
(pivot_multi.round(1))

mean               sum                 median          
month            1    2    3       1       2       3      1    2    3
region                                                               
Africa         4.2  4.1  3.8   506.8   455.5   448.1    3.6  3.6  3.2
Asia           5.7  6.4  5.7  1034.0  1043.2  1037.5    5.3  5.8  5.7
Europe         3.3  3.2  3.1   399.6   349.9   373.0    2.8  2.7  2.3
North America  3.5  3.3  2.8   422.2   355.6   331.1    3.1  2.3  2.0
Oceania        3.9  3.6  4.4   240.4   202.2   240.7    3.8  3.4  3.7
South America  3.3  3.7  3.4   303.5   297.9   307.2    2.9  2.8  2.5

### 5.2 `pivot_table()` กับหลายคอลัมน์ values พร้อมกัน


In [ ]:
pivot_v2 = df.pivot_table(
    index=["city","country"],
    columns="month",
    values=["temperature_c", "humidity_pct", "rainfall_mm"],
    aggfunc="mean"
)
(pivot_v2.round(1).head(10))

humidity_pct             rainfall_mm            \
month                                1     2     3           1    2    3   
city         country                                                       
Auckland     NEW ZEALAND           NaN   NaN  74.6         NaN  NaN  3.5   
             New Zealand          78.5  76.3  77.4         3.5  3.4  5.6   
Bangkok      Thailand             76.0  78.2  76.1         7.5  9.0  7.0   
Beijing      China                45.1  43.7  45.0         2.6  3.1  2.8   
Berlin       GERMANY               NaN   NaN  79.8         NaN  NaN  0.0   
             Germany              70.7  67.7  70.3         3.3  2.7  3.3   
Buenos Aires ARGENTINA            80.3   NaN   NaN         9.3  NaN  NaN   
             Argentina            69.1  67.6  69.9         2.8  4.5  3.0   
Cairo        Egypt                45.4  45.8  46.0         1.0  1.7  1.5   
Cape Town    South Africa         65.7  66.0  64.0         3.2  2.3  2.7   

                          temperature_c              
month                                 1     2     3  
city         country                                 
Auckland     NEW ZEALAND            NaN   NaN  19.8  
             New Zealand           15.9  18.6  18.9  
Bangkok      Thailand              31.2  32.7  33.4  
Beijing      China                 13.5  16.0  17.4  
Berlin       GERMANY                NaN   NaN  14.7  
             Germany               10.7  12.4  14.1  
Buenos Aires ARGENTINA             22.2   NaN   NaN  
             Argentina             15.7  20.6  21.1  
Cairo        Egypt                 24.7  26.3  27.2  
Cape Town    South Africa          18.1  53.6  20.6

### 5.3 จัดการ missing values ใน `pivot_table()` ด้วย `fill_value`

ถ้าบางช่องในตารางไม่มีข้อมูลเลย (เช่น ภูมิภาคนั้นไม่มีข้อมูลเดือนนั้น) จะได้ `NaN` — ใส่ `fill_value` เพื่อแทนที่ด้วยค่าที่ต้องการ


In [ ]:
pivot_filled = df.pivot_table(
    index="region",
    columns="month",
    values="temperature_c",
    aggfunc="mean",
    fill_value=0   # แทน NaN ด้วย 0 (เลือกค่าที่เหมาะกับบริบทข้อมูลเสมอ ไม่ใช่ใส่ 0 ทุกครั้ง)
)
(pivot_filled.round(1))

month,1,2,3
region,,,
Africa,22.6,32.6,25.4
Asia,24.2,26.8,32.7
Europe,10.4,12.2,21.4
North America,15.2,17.1,18.0
Oceania,17.4,19.7,20.2
South America,19.2,22.1,22.3


---
## 6. `merge()` / `join()` — รวม DataFrame ตามคีย์

จนถึงตอนนี้ทำงานกับ `world_weather_sample.csv` ไฟล์เดียว — แต่งานจริงมักมีข้อมูลกระจายอยู่หลายตาราง (เหมือนตารางในฐานข้อมูล) ต้อง **รวม (join)** เข้าด้วยกันก่อนวิเคราะห์

`pd.merge()` คือคำสั่งหลักที่ใช้รวม DataFrame 2 ตัวเข้าด้วยกัน โดยอ้างอิงจาก **คีย์ร่วม** (คอลัมน์ที่มีอยู่ในทั้งสองตาราง) — แนวคิดเดียวกับ `JOIN` ใน SQL

มาดูข้อมูลอีก 2 ตารางที่จะใช้ในหัวข้อนี้: `city_info.csv` (ข้อมูลเมือง สะอาด ไม่มีปัญหา) และ `region_info.csv` (ข้อมูลภูมิภาค)


In [ ]:
city_info = pd.read_csv("city_info.csv")
print("city_info.csv:")
print(city_info)

print()

region_info = pd.read_csv("region_info.csv")
print("region_info.csv:")
print(region_info)

city_info.csv:
            city       country         region  population_millions  timezone
0        Bangkok      Thailand           Asia                10.70     UTC+7
1     Chiang Mai      Thailand           Asia                 0.13     UTC+7
2          Tokyo         Japan           Asia                13.90     UTC+9
3      Singapore     Singapore           Asia                 5.90     UTC+8
4        Beijing         China           Asia                21.50     UTC+8
5         Mumbai         India           Asia                20.40  UTC+5:30
6         London            UK         Europe                 9.00     UTC+0
7          Paris        France         Europe                 2.10     UTC+1
8         Berlin       Germany         Europe                 3.70     UTC+1
9         Moscow        Russia         Europe                12.50     UTC+3
10      New York           USA  North America                 8.30     UTC-5
11   Los Angeles           USA  North America                

### 6.1 `merge()` พื้นฐาน — รวมตามคอลัมน์ที่มีชื่อเดียวกัน

ถ้าทั้งสองตารางมีคอลัมน์ชื่อเดียวกัน (`city`) `merge()` จะใช้คอลัมน์นั้นเป็นคีย์โดยอัตโนมัติ


In [ ]:
# รวมข้อมูลอากาศรายวันกับข้อมูลเมือง (ประชากร, timezone) โดยใช้ city เป็นคีย์
weather_with_city_info = pd.merge(df, city_info[["city", "population_millions", "timezone"]], on="city")
print(weather_with_city_info[["city", "date", "temperature_c", "population_millions", "timezone"]].head())
print(f"\nshape ก่อน merge: {df.shape}, shape หลัง merge: {weather_with_city_info.shape}")

          city       date  temperature_c  population_millions  timezone
0     New York 2025-03-24           15.7                  8.3     UTC-5
1        Tokyo 2025-02-09           18.3                 13.9     UTC+9
2  Los Angeles 2025-02-01           20.6                  3.9     UTC-8
3       Mumbai 2025-01-19           31.0                 20.4  UTC+5:30
4    Cape Town 2025-02-03           19.1                  4.6     UTC+2

shape ก่อน merge: (2075, 10), shape หลัง merge: (2075, 12)


**สังเกต:** จำนวนแถวยังเท่ากับ `df` เดิม (2075 แถว) เพราะ `city_info.csv` มี 1 แถวต่อเมือง และทุกเมืองใน `df` มีอยู่ใน `city_info` ครบ — แต่จำนวน**คอลัมน์**เพิ่มขึ้น เพราะดึงข้อมูล `population_millions` และ `timezone` เข้ามาด้วย

### 6.2 ชนิดของการ Merge: `how` parameter

| `how` | ความหมาย | เปรียบเทียบ SQL |
|---|---|---|
| `"inner"` (default) | เอาเฉพาะแถวที่มีคีย์ตรงกัน**ทั้งสองตาราง** | INNER JOIN |
| `"left"` | เอาทุกแถวจากตารางซ้าย แม้ไม่มีคู่ตรงกันในตารางขวา (เติม NaN) | LEFT JOIN |
| `"right"` | เอาทุกแถวจากตารางขวา | RIGHT JOIN |
| `"outer"` | เอาทุกแถวจากทั้งสองตาราง | FULL OUTER JOIN |


In [ ]:
# ตัวอย่าง: สมมติมีเมืองในข้อมูลอากาศที่ไม่มีอยู่ใน city_info (ทดสอบ how="left")
extra_city = pd.DataFrame([{"city": "Unknown City", "date": "2025-01-01", "country": "?",
                             "region": "?", "temperature_c": 25.0, "humidity_pct": 60.0,
                             "rainfall_mm": 2.0, "wind_speed_kmh": 10.0, "pressure_hpa": 1013.0}])
df_test = pd.concat([df.head(3), extra_city], ignore_index=True)   # ใช้ concat() ต่อ (สอนหัวข้อถัดไป)

# left join: เก็บทุกแถวจาก df_test แม้ไม่มีข้อมูล city_info ของ "Unknown City"
left_result = pd.merge(df_test, city_info[["city", "population_millions"]], on="city", how="left")
print(left_result[["city", "population_millions"]])
print("\nสังเกต: 'Unknown City' ได้ NaN เพราะไม่มีอยู่ใน city_info -- แต่แถวไม่ได้ถูกตัดทิ้ง (เพราะใช้ how='left')")

           city  population_millions
0      New York                  8.3
1         Tokyo                 13.9
2   Los Angeles                  3.9
3  Unknown City                  NaN

สังเกต: 'Unknown City' ได้ NaN เพราะไม่มีอยู่ใน city_info -- แต่แถวไม่ได้ถูกตัดทิ้ง (เพราะใช้ how='left')


### 6.3 `merge()` เมื่อชื่อคอลัมน์คีย์ไม่เหมือนกัน

ถ้าคอลัมน์คีย์ของสองตารางชื่อไม่เหมือนกัน ใช้ `left_on` และ `right_on` ระบุแยกกัน


In [ ]:
# สมมติว่า region_info ใช้ชื่อคอลัมน์ "region" เหมือนกับ df พอดี (กรณีนี้ใช้ on="region" ได้ตรงๆ)
weather_with_region = pd.merge(df.head(5), region_info, on="region")
print(weather_with_region[["city", "region", "continent_code", "avg_gdp_growth_pct"]])

# ตัวอย่างกรณีชื่อคอลัมน์ไม่ตรงกัน (จำลองโดยเปลี่ยนชื่อคอลัมน์ใน region_info)
region_info_renamed = region_info.rename(columns={"region": "zone_name"})
weather_with_region_v2 = pd.merge(
    df.head(5), region_info_renamed,
    left_on="region", right_on="zone_name"   # ระบุชื่อคอลัมน์คีย์ของแต่ละตารางแยกกัน
)
print("\nกรณีชื่อคอลัมน์คีย์ไม่ตรงกัน (ใช้ left_on/right_on):")
print(weather_with_region_v2[["city", "region", "zone_name", "continent_code"]])

### 6.4 รวมหลายตารางต่อกันเป็นทอด (Chained Merge)

ในงานจริงมักต้อง merge มากกว่า 2 ตาราง — ทำได้โดย merge ทีละคู่ต่อกันเป็นทอดๆ


In [ ]:
# Merge 3 ตารางเข้าด้วยกัน: weather -> city_info -> region_info
full_data = (
    df
    .merge(city_info[["city", "population_millions", "timezone"]], on="city", how="left")
    .merge(region_info, on="region", how="left")
)
print(full_data[["city", "region", "population_millions", "continent_code", "avg_gdp_growth_pct"]].head())
print(f"\nshape สุดท้าย: {full_data.shape}")

### 6.5 `.join()` — ทางเลือกที่สั้นกว่าเมื่อรวมตาม index

`.join()` คล้าย `.merge()` มาก แต่ออกแบบมาให้รวมตาม **index** เป็นหลัก (ไม่ใช่คอลัมน์) — สั้นกว่าเมื่อข้อมูลทั้งสองตารางใช้ index เดียวกันอยู่แล้ว


In [ ]:
city_info_indexed = city_info.set_index("city")[["population_millions", "timezone"]]
df_indexed = df.set_index("city")

joined = df_indexed.join(city_info_indexed)
print(joined[["date", "temperature_c", "population_millions", "timezone"]].head())

---
## 7. `concat()` — ต่อ DataFrame เข้าด้วยกัน

`pd.concat()` ใช้ "ต่อ" DataFrame หลายตัวเข้าด้วยกัน — **ต่างจาก `merge()`** ที่รวมตามคีย์ `concat()` แค่ "เอามาเรียงต่อกัน" เฉยๆ (เหมือนเอา list 2 ตัวมาต่อกันด้วย `+` ที่เรียนมา)

### 7.1 `concat()` แนวตั้ง (ต่อแถว) — กรณีใช้บ่อยที่สุด

ใช้เมื่อมีข้อมูลรูปแบบเดียวกันแต่มาจากหลายไฟล์/หลายช่วงเวลา ต้องการเอามารวมเป็นตารางเดียว


In [ ]:
# จำลองข้อมูล 2 ช่วงเวลาที่แยกไฟล์กัน (เช่น ข้อมูลเดือน 1 กับเดือน 2 อยู่คนละไฟล์)
batch1 = df[df["month"] == 1]
batch2 = df[df["month"] == 2]

print(f"batch1: {batch1.shape}, batch2: {batch2.shape}")

combined = pd.concat([batch1, batch2], ignore_index=True)   # ignore_index=True สร้าง index ใหม่ให้ต่อเนื่อง
print(f"combined: {combined.shape}")
print(combined["month"].value_counts())

**⚠️ ข้อสำคัญ:** ควรใส่ `ignore_index=True` เสมอเมื่อ concat แนวตั้ง ไม่อย่างนั้น index จะซ้ำกัน (เช่น มี index 0 ซ้ำกัน 2 รอบจากทั้งสองตาราง) ทำให้ `.loc[]` สับสนได้

### 7.2 `concat()` แนวนอน (ต่อคอลัมน์)

ใช้ `axis=1` เมื่อต้องการต่อ DataFrame ทาง "ข้าง" (เพิ่มคอลัมน์) แทนทาง "ล่าง" (เพิ่มแถว) — ต้องระมัดระวังเรื่อง index ให้ตรงกัน ไม่อย่างนั้นข้อมูลจะเรียงผิดแถว


In [ ]:
# ตัวอย่าง concat แนวนอน - ต้อง index ตรงกันก่อน
temp_data = df[["city", "temperature_c"]].head(3)
humidity_data = df[["humidity_pct"]].head(3)

horizontal_concat = pd.concat([temp_data, humidity_data], axis=1)
print(horizontal_concat)

          city  temperature_c  humidity_pct
0     New York           15.7          72.6
1        Tokyo           18.3          60.6
2  Los Angeles           20.6          46.8


### 7.3 `concat()` vs `merge()` — สรุปเมื่อไหร่ใช้อันไหน

| | `concat()` | `merge()` |
|---|---|---|
| ใช้เมื่อ | ข้อมูลรูปแบบเดียวกัน อยากต่อกันตรงๆ | ข้อมูลคนละตาราง มีคีย์ร่วม อยากรวมตามคีย์นั้น |
| อ้างอิงจาก | ตำแหน่ง/index | คอลัมน์คีย์ (ค่าต้องตรงกันจึงรวมกัน) |
| ตัวอย่างการใช้งาน | รวมไฟล์ข้อมูลเดือน 1 + เดือน 2 | รวมข้อมูลอากาศกับข้อมูลเมือง |


---
## 8. `apply()` + `lambda` — เขียนฟังก์ชันกำหนดเองไปใช้กับข้อมูล

บางครั้งไม่มีฟังก์ชันสำเร็จรูปของ pandas ที่ตรงกับสิ่งที่อยากทำ — `.apply()` ให้นำ**ฟังก์ชันของตัวเอง**ไปใช้กับทุกแถว/ทุกค่าใน Series หรือ DataFrame ได้ เชื่อมตรงกับเรื่อง **function** (`def`) ที่เรียนมา

### 8.1 `apply()` กับฟังก์ชันที่นิยามด้วย `def`


In [ ]:
def classify_temperature(temp):
    if temp >= 35:
        return "ร้อนจัด"
    elif temp >= 25:
        return "ร้อน"
    elif temp >= 15:
        return "อบอุ่น"
    else:
        return "เย็น"

df["temp_category"] = df["temperature_c"].apply(classify_temperature)
print(df[["city", "temperature_c", "temp_category"]].head(10))

**สังเกต:** `classify_temperature` คือฟังก์ชันธรรมดาที่เรียนมาจาก `def` — `.apply()` เอาฟังก์ชันนี้ไปรันกับ**ทุกค่า**ในคอลัมน์ `temperature_c` แล้วเก็บผลลัพธ์เป็นคอลัมน์ใหม่ เทียบเท่ากับการเขียน `for` loop วนทุกแถวแล้วเรียกฟังก์ชันเอง แต่สั้นและเร็วกว่า

### 8.2 `lambda` — ฟังก์ชันแบบสั้นในบรรทัดเดียว

**`lambda`** คือวิธีเขียนฟังก์ชันแบบย่อ ใช้เมื่อฟังก์ชัน**ง่ายมาก**และใช้แค่ครั้งเดียว ไม่จำเป็นต้องตั้งชื่อด้วย `def`

```python
lambda parameter: expression
```

เทียบเท่ากับ:
```python
def some_name(parameter):
    return expression
```


In [ ]:
# ฟังก์ชันง่ายๆ ที่แปลงองศาเซลเซียสเป็นฟาเรนไฮต์

# วิธีปกติด้วย def
def celsius_to_fahrenheit(c):
    return c * 9/5 + 32

# วิธี lambda (ใช้ครั้งเดียว ไม่ตั้งชื่อ)
df["temp_fahrenheit"] = df["temperature_c"].apply(lambda c: c * 9/5 + 32)
print(df[["city", "temperature_c", "temp_fahrenheit"]].head())

# ตรวจสอบว่าได้ผลลัพธ์เหมือนกับใช้ def
df["temp_fahrenheit_v2"] = df["temperature_c"].apply(celsius_to_fahrenheit)
print("\nผลลัพธ์เหมือนกันไหม:", df["temp_fahrenheit"].equals(df["temp_fahrenheit_v2"]))

### 8.3 `lambda` กับเงื่อนไข (conditional expression)

`lambda` เขียน `if-else` ได้ในบรรทัดเดียว ใช้รูปแบบ `ค่าถ้าจริง if เงื่อนไข else ค่าถ้าเท็จ` (เคยเห็นรูปแบบนี้มาแล้วตอนเรียน comprehension)


In [ ]:
# ใช้ lambda กับเงื่อนไขง่ายๆ (ไม่ต้องตั้งชื่อฟังก์ชันแยกแบบ classify_temperature)
df["is_rainy"] = df["rainfall_mm"].apply(lambda x: "ฝนตก" if x > 5 else "ไม่ตก")
print(df[["city", "rainfall_mm", "is_rainy"]].head(10))

### 8.4 `apply()` กับหลายคอลัมน์พร้อมกัน (ใช้ `axis=1`)

ถ้าฟังก์ชันต้องใช้ค่าจาก**หลายคอลัมน์**พร้อมกัน ใช้ `axis=1` แล้วรับ "แถวทั้งแถว" เข้าฟังก์ชัน (เข้าถึงแต่ละคอลัมน์เหมือน dict ที่เรียนมา)


In [ ]:
def comfort_index(row):
    # ฟังก์ชันจำลองดัชนีความสบาย จากอุณหภูมิและความชื้นร่วมกัน
    if row["temperature_c"] > 30 and row["humidity_pct"] > 70:
        return "อบอ้าว"
    elif row["temperature_c"] < 15:
        return "หนาว"
    else:
        return "สบาย"

df["comfort"] = df.apply(comfort_index, axis=1)   # axis=1 -> ส่งทั้งแถวเข้าฟังก์ชัน
print(df[["city", "temperature_c", "humidity_pct", "comfort"]].head(10))
print("\nสรุปจำนวนแต่ละระดับความสบาย:")
print(df["comfort"].value_counts())

### 8.5 เมื่อไหร่ใช้ `apply()`/`lambda` และเมื่อไหร่ไม่ควร

**ควรใช้เมื่อ:** ไม่มีฟังก์ชันสำเร็จรูปของ pandas ที่ตรงกับ logic ที่ต้องการ (เช่น เงื่อนไขซับซ้อนที่ผสมหลายคอลัมน์)

**ไม่ควรใช้เมื่อ:** มีฟังก์ชัน vectorized ของ pandas/NumPy ที่ทำสิ่งเดียวกันได้อยู่แล้ว — `.apply()` ทำงานทีละแถว (ช้ากว่า) ในขณะที่ฟังก์ชัน vectorized ทำงานกับทั้งคอลัมน์พร้อมกัน (เร็วกว่ามาก เหมือนที่เรียนมาจาก NumPy)

```python
# ❌ ไม่ควรทำ - ใช้ apply() กับงานที่ vectorized ทำได้อยู่แล้ว (ช้ากว่าโดยไม่จำเป็น)
df["temp_fahrenheit"] = df["temperature_c"].apply(lambda c: c * 9/5 + 32)

# ✅ ควรทำ - ใช้ vectorized operation ตรงๆ (เร็วกว่า อ่านง่ายกว่า)
df["temp_fahrenheit"] = df["temperature_c"] * 9/5 + 32
```


In [ ]:
# เปรียบเทียบผลลัพธ์ว่าเหมือนกัน แต่วิธี vectorized เร็วกว่า
df["temp_fahrenheit_vectorized"] = df["temperature_c"] * 9/5 + 32
print("ผลลัพธ์เหมือนกับ apply() ไหม:", df["temp_fahrenheit"].equals(df["temp_fahrenheit_vectorized"]))

---
## 🧪 แบบฝึกหัดท้ายคาบ

> ลองทำเองในเซลล์ที่เตรียมไว้ด้านล่างแต่ละข้อ ไม่มีเฉลยให้ — ถ้าไม่แน่ใจให้ลองรันดูผลลัพธ์ หรือถามอาจารย์/เพื่อนได้เลย ใช้ไฟล์ `world_weather_sample.csv`, `city_info.csv`, `region_info.csv`

### ข้อ 1: groupby และ agg
1. หาอุณหภูมิเฉลี่ย ความชื้นเฉลี่ย และปริมาณฝนรวม แยกตาม `region` (ใช้ `.agg()` พร้อมตั้งชื่อคอลัมน์เอง)
2. เรียงผลลัพธ์ตามอุณหภูมิเฉลี่ยจากมากไปน้อย


In [ ]:
# เขียนคำตอบข้อ 1 ที่นี่


In [5]:
region_summary = df.groupby("region").agg(
    avg_temperature=("temperature_c", "mean"),
    avg_humidity=("humidity_pct", "mean"),
    total_rainfall=("rainfall_mm", "sum")
)

# เรียงผลลัพธ์ตามอุณหภูมิเฉลี่ยจากมากไปน้อย
sorted_region_summary = region_summary.sort_values(by="avg_temperature", ascending=False)
display(sorted_region_summary)

,avg_temperature,avg_humidity,total_rainfall
region,,,
Asia,27.920223,66.155641,3114.7
Africa,26.766479,63.613372,1410.4
South America,21.169030,71.692720,908.6
Oceania,19.076111,71.234104,683.3
North America,16.751541,60.379825,1108.9
Europe,14.712079,71.431714,1122.5


### ข้อ 2: value_counts และ sort_values
1. หาว่าเมืองไหนมีจำนวนวันที่ฝนตกหนัก (`rainfall_mm > 10`) มากที่สุด 5 อันดับแรก (Hint: กรองด้วย boolean indexing ก่อน แล้วใช้ `value_counts()` กับคอลัมน์ city)
2. ใช้ `.nlargest()` หา 5 วันที่ลมแรงที่สุดในข้อมูลทั้งหมด


In [7]:
# เขียนคำตอบข้อ 2 ที่นี่
# 1. หาว่าเมืองไหนมีจำนวนวันที่ฝนตกหนัก (rainfall_mm > 10) มากที่สุด 5 อันดับแรก
heavy_rain_days = df[df["rainfall_mm"] > 10]
top_5_cities_heavy_rain = heavy_rain_days["city"].value_counts().nlargest(5)
print("เมืองที่มีจำนวนวันที่ฝนตกหนักที่สุด 5 อันดับแรก:")
display(top_5_cities_heavy_rain)

เมืองที่มีจำนวนวันที่ฝนตกหนักที่สุด 5 อันดับแรก:


,count
city,
Singapore,44
Bangkok,23
Lagos,22
Sao Paulo,11
Buenos Aires,9


In [8]:
# 2. ใช้ .nlargest() หา 5 วันที่ลมแรงที่สุดในข้อมูลทั้งหมด
top_5_windy_days = df.nlargest(5, "wind_speed_kmh")
print("\n5 วันที่ลมแรงที่สุดในข้อมูลทั้งหมด:")
display(top_5_windy_days[["city", "date", "wind_speed_kmh"]])


5 วันที่ลมแรงที่สุดในข้อมูลทั้งหมด:


,city,date,wind_speed_kmh
357,Lima,2025-01-05,33.7
1823,Chiang Mai,2025-01-06,33.5
587,Sao Paulo,2025-03-29,32.6
1735,Mexico City,2025-02-24,31.9
1974,Cape Town,2025-02-21,31.6


### ข้อ 3: pivot_table
สร้าง pivot table ที่มี:
- index = `city`
- columns = `month`
- values = `rainfall_mm`
- aggfunc = `"sum"` (ผลรวมฝนตกแต่ละเดือนของแต่ละเมือง)

จากนั้นหาว่าเมืองไหนมีฝนตกรวมทั้ง 3 เดือนมากที่สุด (Hint: ใช้ `.sum(axis=1)` กับผลลัพธ์ pivot table เพื่อรวมทุกเดือน)


In [11]:
# เขียนคำตอบข้อ 3 ที่นี่
# สร้างคอลัมน์เดือนจาก date (จากตัวอย่างด้านบน)
df["month"] = df["date"].dt.month

# 1. สร้าง pivot table
pivot_rainfall = df.pivot_table(
    index="city",
    columns="month",
    values="rainfall_mm",
    aggfunc="sum",
    fill_value=0 # แทน NaN ด้วย 0 เพื่อให้รวมผลได้ถูกต้อง
)
display(pivot_rainfall.round(1))

# 2. หาเมืองที่มีฝนตกรวมทั้ง 3 เดือนมากที่สุด
total_rainfall_by_city = pivot_rainfall.sum(axis=1)
max_rainfall_city = total_rainfall_by_city.idxmax()
max_rainfall_value = total_rainfall_by_city.max()

print(f"\nเมืองที่มีฝนตกรวมทั้ง 3 เดือนมากที่สุดคือ {max_rainfall_city} ด้วยปริมาณฝนรวม {max_rainfall_value:.1f} มม.")
display(total_rainfall_by_city.sort_values(ascending=False).head())

month,1,2,3
city,,,
Auckland,106.2,95.7,150.0
Bangkok,231.6,243.9,210.1
Beijing,77.6,87.4,85.1
Berlin,98.4,69.9,100.1
Buenos Aires,93.5,121.2,90.4
Cairo,29.0,48.0,46.0
Cape Town,98.3,62.3,78.7
Chiang Mai,109.8,137.8,188.0
Lagos,201.3,218.5,233.0



เมืองที่มีฝนตกรวมทั้ง 3 เดือนมากที่สุดคือ Singapore ด้วยปริมาณฝนรวม 870.4 มม.


,0
city,
Singapore,870.4
Bangkok,685.6
Lagos,652.8
Mumbai,487.9
Chiang Mai,435.6


### ข้อ 4: merge
1. Merge `world_weather_sample.csv` กับ `city_info.csv` โดยใช้ `city` เป็นคีย์ (เลือกใช้ `how` ที่เหมาะสม)
2. จากผลลัพธ์ที่ merge แล้ว ใช้ `groupby()` หาอุณหภูมิเฉลี่ยแยกตาม `timezone`
3. Merge ผลลัพธ์ที่ได้เพิ่มกับ `region_info.csv` ด้วย (merge 3 ตารางต่อกันเป็นทอด) แล้วแสดงคอลัมน์ `city`, `region`, `continent_code`, `population_millions` ของ 10 แถวแรก


In [ ]:
# เขียนคำตอบข้อ 4 ที่นี่


In [28]:
# Load city_info and region_info DataFrames first (if not already loaded)
city_info = pd.read_csv("city_info.csv")
region_info = pd.read_csv("region_info.csv")

# 1. Merge world_weather_sample.csv (df) กับ city_info.csv
# ใช้ 'left' merge เพื่อเก็บข้อมูลสภาพอากาศทั้งหมดและเพิ่มข้อมูลเมือง
# Select only necessary columns from city_info to avoid 'region' column conflict
df_with_city_info = pd.merge(df, city_info[["city", "population_millions", "timezone"]], on="city", how="left")

# 2. จากผลลัพธ์ที่ merge แล้ว ใช้ groupby() หาอุณหภูมิเฉลี่ยแยกตาม timezone
avg_temp_by_timezone = df_with_city_info.groupby("timezone")["temperature_c"].mean().reset_index()
print("\nอุณหภูมิเฉลี่ยแยกตาม Timezone:")
display(avg_temp_by_timezone)

# 3. Merge ผลลัพธ์ที่ได้ (df_with_city_info) เพิ่มกับ region_info.csv
# ใช้ 'left' merge เพื่อเพิ่มข้อมูลภูมิภาค. 'region' column is now correctly named after the first merge.
final_merged_df = pd.merge(df_with_city_info, region_info, on="region", how="left")

# แสดงคอลัมน์ city, region, continent_code, population_millions ของ 10 แถวแรก
print("\n10 แถวแรกของข้อมูลที่รวมกันแล้ว:")
display(final_merged_df[["city", "region", "continent_code", "population_millions"]].head(10))


อุณหภูมิเฉลี่ยแยกตาม Timezone:


,timezone,temperature_c
0,UTC+0,13.490909
1,UTC+1,22.510566
2,UTC+10,20.347778
3,UTC+12,17.804444
4,UTC+2,28.126404
5,UTC+3,14.336667
6,UTC+5:30,30.872222
7,UTC+7,31.067033
8,UTC+8,27.909497
9,UTC+9,18.414773



10 แถวแรกของข้อมูลที่รวมกันแล้ว:


,city,region,continent_code,population_millions
0,New York,North America,NAM,8.3
1,Tokyo,Asia,AS,13.9
2,Los Angeles,North America,NAM,3.9
3,Mumbai,Asia,AS,20.4
4,Cape Town,Africa,AF,4.6
5,Auckland,Oceania,OC,1.7
6,Buenos Aires,South America,SAM,3.1
7,Mexico City,North America,NAM,9.2
8,Buenos Aires,South America,SAM,3.1
9,Los Angeles,North America,NAM,3.9


### ข้อ 5: concat
1. แบ่งข้อมูล `world_weather_sample.csv` เป็น 2 ส่วนตาม `region` (เช่น `region == "Asia"` กับ `region != "Asia"`)
2. ใช้ `concat()` ต่อทั้ง 2 ส่วนกลับเข้าด้วยกัน (ใส่ `ignore_index=True`)
3. ตรวจสอบว่าจำนวนแถวหลัง concat เท่ากับจำนวนแถวต้นฉบับหรือไม่


In [30]:
# เขียนคำตอบข้อ 5 ที่นี่
asia_df = df[df["region"] == "Asia"]
other_regions_df = df[df["region"] != "Asia"]

# 2. ใช้ concat() ต่อทั้ง 2 ส่วนกลับเข้าด้วยกัน (ใส่ ignore_index=True)
combined_df = pd.concat([asia_df, other_regions_df], ignore_index=True)

# 3. ตรวจสอบว่าจำนวนแถวหลัง concat เท่ากับจำนวนแถวต้นฉบับหรือไม่
original_rows = df.shape[0]
combined_rows = combined_df.shape[0]

print(f"จำนวนแถวของ DataFrame ต้นฉบับ: {original_rows}")
print(f"จำนวนแถวของ DataFrame หลัง concat: {combined_rows}")

if original_rows == combined_rows:
    print("✅ จำนวนแถวหลัง concat เท่ากับจำนวนแถวต้นฉบับ")
else:
    print("❌ จำนวนแถวหลัง concat ไม่เท่ากับจำนวนแถวต้นฉบับ")

# แสดงหัวตารางของ combined_df เพื่อยืนยันว่ามีข้อมูลทั้งจาก Asia และภูมิภาคอื่นๆ
display(combined_df.head())

จำนวนแถวของ DataFrame ต้นฉบับ: 2075
จำนวนแถวของ DataFrame หลัง concat: 2075
✅ จำนวนแถวหลัง concat เท่ากับจำนวนแถวต้นฉบับ


,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa,month
0,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1,2
1,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1,1
2,434,2025-03-15,Beijing,China,Asia,14.3,38.5,0.0,13.6,1015.8,3
3,357,2025-03-28,Singapore,Singapore,Asia,28.5,79.3,3.0,19.6,1013.5,3
4,334,2025-03-05,Singapore,Singapore,Asia,31.5,150.0,7.0,20.0,1016.3,3


### ข้อ 6: apply + lambda
1. เขียนฟังก์ชัน (ด้วย `def`) ชื่อ `wind_category(speed)` ที่แปลงความเร็วลมเป็นหมวดหมู่: `< 10` = "ลมเบา", `10-20` = "ลมปานกลาง", `> 20` = "ลมแรง" แล้วใช้ `.apply()` สร้างคอลัมน์ใหม่
2. ใช้ `lambda` สร้างคอลัมน์ใหม่ที่บอกว่าอุณหภูมิสูงกว่า 30°C หรือไม่ (True/False)
3. ใช้ `.apply(axis=1)` เขียนฟังก์ชันที่รวมเงื่อนไขจาก `temperature_c` และ `wind_speed_kmh` เพื่อจัดหมวดหมู่ "สภาพอากาศที่ควรเลี่ยงกิจกรรมนอกบ้าน" (กำหนดเกณฑ์เองได้)


In [43]:
# 1. เขียนฟังก์ชัน (ด้วย def) ชื่อ wind_category(speed) ที่แปลงความเร็วลมเป็นหมวดหมู่
def wind_category(speed):
    if speed < 10:
        return "ลมเบา"
    elif 10 <= speed <= 20:
        return "ลมปานกลาง"
    else:
        return "ลมแรง"

# ใช้ .apply() สร้างคอลัมน์ใหม่ 'wind_category'
df["wind_category"] = df["wind_speed_kmh"].apply(wind_category)
print("5 แถวแรกของคอลัมน์ 'wind_speed_kmh' และ 'wind_category':")
display(df[["wind_speed_kmh", "wind_category"]].head())

# 2. ใช้ lambda สร้างคอลัมน์ใหม่ที่บอกว่าอุณหภูมิสูงกว่า 30°C หรือไม่ (True/False)
df["is_hot_temp"] = df["temperature_c"].apply(lambda x: x > 30)
print("\n5 แถวแรกของคอลัมน์ 'temperature_c' และ 'is_hot_temp':")
display(df[["temperature_c", "is_hot_temp"]].head())

# 3. ใช้ .apply(axis=1) เขียนฟังก์ชันที่รวมเงื่อนไขจาก temperature_c และ wind_speed_kmh
# เพื่อจัดหมวดหมู่ "สภาพอากาศที่ควรเลี่ยงกิจกรรมนอกบ้าน" (กำหนดเกณฑ์เองได้)
def outdoor_warning(row):
    temp = row["temperature_c"]
    wind = row["wind_speed_kmh"]

    if temp > 35 and wind > 25: # ร้อนจัดและลมแรงมาก
        return "ควรเลี่ยงกิจกรรมกลางแจ้ง (ร้อนจัดและลมแรง)"
    elif temp < 5 and wind > 20: # หนาวจัดและลมแรง
        return "ควรเลี่ยงกิจกรรมกลางแจ้ง (หนาวจัดและลมแรง)"
    elif temp > 30 and wind > 15: # ร้อนและลมแรงปานกลาง
        return "ระวัง (ร้อนและลมแรง)"
    elif temp < 10 and wind > 10: # เย็นและลมปานกลาง
        return "อากาศเย็น มีลม"
    else:
        return "ปกติ (เหมาะสำหรับกิจกรรมนอกบ้าน)"

df["outdoor_warning"] = df.apply(outdoor_warning, axis=1)
print("\n5 แถวแรกของคอลัมน์ 'temperature_c', 'wind_speed_kmh' และ 'outdoor_warning':")
display(df[["temperature_c", "wind_speed_kmh", "outdoor_warning"]].head())

5 แถวแรกของคอลัมน์ 'wind_speed_kmh' และ 'wind_category':


,wind_speed_kmh,wind_category
0,15.9,ลมปานกลาง
1,20.5,ลมแรง
2,8.6,ลมเบา
3,11.8,ลมปานกลาง
4,16.6,ลมปานกลาง



5 แถวแรกของคอลัมน์ 'temperature_c' และ 'is_hot_temp':


,temperature_c,is_hot_temp
0,15.7,False
1,18.3,False
2,20.6,False
3,31.0,True
4,19.1,False



5 แถวแรกของคอลัมน์ 'temperature_c', 'wind_speed_kmh' และ 'outdoor_warning':


,temperature_c,wind_speed_kmh,outdoor_warning
0,15.7,15.9,ปกติ (เหมาะสำหรับกิจกรรมนอกบ้าน)
1,18.3,20.5,ปกติ (เหมาะสำหรับกิจกรรมนอกบ้าน)
2,20.6,8.6,ปกติ (เหมาะสำหรับกิจกรรมนอกบ้าน)
3,31.0,11.8,ปกติ (เหมาะสำหรับกิจกรรมนอกบ้าน)
4,19.1,16.6,ปกติ (เหมาะสำหรับกิจกรรมนอกบ้าน)


##ข้อ 7: ท้าทายเพิ่มเติม (Challenge) — รวมทุกอย่างเข้าด้วยกัน
จงเขียนโค้ดที่:
1. Merge ข้อมูลอากาศกับ city_info.csv และ region_info.csv (3 ตารางรวมกัน)
2. ใช้ apply()/lambda สร้างคอลัมน์ใหม่ temp_category (เย็น/อบอุ่น/ร้อน/ร้อนจัด ตามเกณฑ์ที่กำหนดเอง)
3. ใช้ groupby() กับ ["region", "temp_category"] แล้ว .agg() หาจำนวนวันและอุณหภูมิเฉลี่ยของแต่ละกลุ่ม
4. ใช้ pivot_table() แปลงผลลัพธ์ข้อ 3 ให้เป็นตาราง: แถว = region, คอลัมน์ = temp_category, ค่า = จำนวนวัน
5. เรียงและแสดงผลลัพธ์สุดท้ายให้อ่านง่าย


In [38]:
# Load city_info and region_info DataFrames (if not already loaded)
city_info = pd.read_csv("city_info.csv")
region_info = pd.read_csv("region_info.csv")

# 1. Merge ข้อมูลอากาศกับ city_info.csv และ region_info.csv (3 ตารางรวมกัน)
# สร้าง DataFrame ใหม่เพื่อไม่ให้กระทบ df เดิมที่มีคอลัมน์จากข้อ 6 แล้ว
merged_df_challenge = (
    df.drop(columns=["wind_category", "is_hot_temp", "outdoor_warning"])
    .merge(city_info[["city", "population_millions", "timezone"]], on="city", how="left")
    .merge(region_info, on="region", how="left")
)

print("ข้อมูลหลัง Merge 3 ตาราง (5 แถวแรก):")
display(merged_df_challenge.head())
print(f"\nShape หลัง merge: {merged_df_challenge.shape}")

ข้อมูลหลัง Merge 3 ตาราง (5 แถวแรก):


,record_id,date,city,country,region,temperature_c,humidity_pct,rainfall_mm,wind_speed_kmh,pressure_hpa,month,population_millions,timezone,continent_code,avg_gdp_growth_pct
0,983,2025-03-24,New York,USA,North America,15.7,72.6,6.3,15.9,1005.4,3,8.3,UTC-5,NAM,2.3
1,220,2025-02-09,Tokyo,Japan,Asia,18.3,60.6,0.0,20.5,1015.1,2,13.9,UTC+9,AS,4.2
2,1022,2025-02-01,Los Angeles,USA,North America,20.6,46.8,1.2,8.6,1013.5,2,3.9,UTC-8,NAM,2.3
3,469,2025-01-19,Mumbai,India,Asia,31.0,73.4,8.0,11.8,1000.1,1,20.4,UTC+5:30,AS,4.2
4,1834,2025-02-03,Cape Town,South Africa,Africa,19.1,55.0,0.0,16.6,1015.1,2,4.6,UTC+2,AF,3.6



Shape หลัง merge: (2075, 15)


In [39]:
# 2. ใช้ apply()/lambda สร้างคอลัมน์ใหม่ temp_category
def classify_temperature_challenge(temp):
    if temp >= 35:
        return "ร้อนจัด"
    elif temp >= 25:
        return "ร้อน"
    elif temp >= 15:
        return "อบอุ่น"
    else:
        return "เย็น"

merged_df_challenge["temp_category"] = merged_df_challenge["temperature_c"].apply(classify_temperature_challenge)

print("5 แถวแรกพร้อมคอลัมน์ 'temp_category':")
display(merged_df_challenge[["temperature_c", "temp_category"]].head())

5 แถวแรกพร้อมคอลัมน์ 'temp_category':


,temperature_c,temp_category
0,15.7,อบอุ่น
1,18.3,อบอุ่น
2,20.6,อบอุ่น
3,31.0,ร้อน
4,19.1,อบอุ่น


In [40]:
# 3. ใช้ groupby() กับ ["region", "temp_category"] แล้ว agg() หาจำนวนวันและอุณหภูมิเฉลี่ยของแต่ละกลุ่ม
region_temp_summary = merged_df_challenge.groupby(["region", "temp_category"]).agg(
    days_count=("record_id", "count"), # ใช้ record_id ในการนับจำนวนวัน
    avg_temperature=("temperature_c", "mean")
).reset_index()

print("สรุปจำนวนวันและอุณหภูมิเฉลี่ยแยกตามภูมิภาคและหมวดหมู่อุณหภูมิ:")
display(region_temp_summary.sort_values(by=["region", "days_count"], ascending=[True, False]))

สรุปจำนวนวันและอุณหภูมิเฉลี่ยแยกตามภูมิภาคและหมวดหมู่อุณหภูมิ:


,region,temp_category,days_count,avg_temperature
2,Africa,อบอุ่น,199,20.752764
0,Africa,ร้อน,146,28.629452
3,Africa,เย็น,13,14.166667
1,Africa,ร้อนจัด,4,276.850000
4,Asia,ร้อน,332,30.653012
6,Asia,อบอุ่น,142,18.390845
7,Asia,เย็น,44,10.353659
5,Asia,ร้อนจัด,24,76.508333
10,Europe,เย็น,279,10.414599
9,Europe,อบอุ่น,81,17.097531


In [41]:
# 4. ใช้ pivot_table() แปลงผลลัพธ์ข้อ 3 ให้เป็นตาราง: แถว = region, คอลัมน์ = temp_category, ค่า = จำนวนวัน
pivoted_challenge = region_temp_summary.pivot_table(
    index="region",
    columns="temp_category",
    values="days_count",
    fill_value=0 # ใส่ 0 ในช่องที่ไม่มีข้อมูล
)

# 5. เรียงและแสดงผลลัพธ์สุดท้ายให้อ่านง่าย
# จัดเรียงคอลัมน์ให้เป็นลำดับที่เหมาะสม (เย็น -> อบอุ่น -> ร้อน -> ร้อนจัด)
category_order = ["เย็น", "อบอุ่น", "ร้อน", "ร้อนจัด"]
pivoted_challenge = pivoted_challenge[category_order]

print("\nตาราง Pivot แสดงจำนวนวันของแต่ละหมวดหมู่อุณหภูมิแยกตามภูมิภาค:")
display(pivoted_challenge)


ตาราง Pivot แสดงจำนวนวันของแต่ละหมวดหมู่อุณหภูมิแยกตามภูมิภาค:


temp_category,เย็น,อบอุ่น,ร้อน,ร้อนจัด
region,,,,
Africa,13.0,199.0,146.0,4.0
Asia,44.0,142.0,332.0,24.0
Europe,279.0,81.0,0.0,1.0
North America,126.0,222.0,12.0,0.0
Oceania,17.0,159.0,4.0,0.0
South America,8.0,224.0,38.0,0.0


---
## 🔗 เชื่อม Colab กับ GitHub

เก็บ Notebook นี้ (พร้อมไฟล์ CSV ทั้ง 3) ขึ้น GitHub ต่อจากไฟล์ก่อนหน้า

### วิธีที่ 1: เซฟจาก Colab ขึ้น GitHub ตรงๆ (สำหรับ Notebook)

1. ใน Colab ไปที่เมนู **File → Save a copy in GitHub**
2. เลือก repository เดิมที่ใช้เก็บ Notebook คาบก่อนๆ (เช่น `SC612104-coursework`)
3. ตั้งชื่อไฟล์และ commit message (เช่น `"เพิ่ม notebook: groupby, merge, pivot_table"`) แล้วกด **OK**

**⚠️ ข้อควรรู้:** วิธีนี้ push แค่ตัว `.ipynb` — ไฟล์ CSV ต้องอัปโหลดขึ้น GitHub แยกต่างหาก

### วิธีที่ 2: ใช้ Git ผ่าน Terminal (push ได้ทั้ง Notebook และ CSV ในคำสั่งเดียว)

```bash
git clone https://github.com/<username>/<repo-name>.git
cd <repo-name>
git add notebook.ipynb world_weather_sample.csv city_info.csv region_info.csv
git commit -m "เพิ่ม notebook และข้อมูล: pandas groupby/merge/pivot"
git push origin main
```